In [ ]:
import os, zipfile, itertools, gdown, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
import timm
import torch.utils.checkpoint as checkpoint


# ╔═══════════════════════════════════════════════════════════════════════╗
# 1.  DEVICE                                                              ║
# ╚═══════════════════════════════════════════════════════════════════════╝

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Running on: {device}")

✅ Running on: cuda


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 2.  DOWNLOAD / PREPARE TEST DATASET                                     ║
# ╚═══════════════════════════════════════════════════════════════════════╝
TEST_FILE_ID = "Your Drive ID"          
TEST_ZIP     = "test_dataset.zip"
TEST_FOLDER  = "new_test_dataset_folder"

if not os.path.exists(TEST_FOLDER):
    print("📥 Downloading test images …")
    gdown.download(f"https://drive.google.com/uc?id={TEST_FILE_ID}", TEST_ZIP, quiet=False)

    print("📦 Extracting …")
    with zipfile.ZipFile(TEST_ZIP) as zf:
        zf.extractall(TEST_FOLDER)
    print(f"✅ Extracted to: {TEST_FOLDER}\n")
else:
    print(f"✅ Test dataset already present: {TEST_FOLDER}\n")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 3.  LABEL MAP (CUIs)                                                    ║
# ╚═══════════════════════════════════════════════════════════════════════╝
DENSE_CKPT = "your_model_checkpoint.pth"
_dense_ckpt = torch.load(DENSE_CKPT, map_location="cpu",weights_only= False)
CLASS_LIST  = _dense_ckpt["class_names"]
mlb = MultiLabelBinarizer(classes=CLASS_LIST)
mlb.fit(CLASS_LIST)
NUM_C = len(CLASS_LIST)
print(f"📋 Number of classes (CUIs): {NUM_C}\n")

📋 Number of classes (CUIs): 2479



In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 4.  MODEL DEFINITIONS                                                   ║
# ╚═══════════════════════════════════════════════════════════════════════╝
class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        if x.dim() == 2:
            return x
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1. / self.p)

class EfficientNetB0WithFFNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.efficientnet = models.efficientnet_b0(weights=None)
        self.efficientnet.classifier = nn.Identity()
        self.gem = GeMPooling()
        self.ffnn = nn.Sequential(   
            nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes), nn.Sigmoid()
        )

    def forward(self, x):
        x = self.efficientnet(x)
        if x.dim() == 2:
            x = x[:, :, None, None]
        x = self.gem(x).flatten(1)
        return self.ffnn(x)          


def build_densenet(n_cls: int):
    m = models.densenet121(weights=None)
    m.classifier = nn.Linear(m.classifier.in_features, n_cls)
    return m

class KCLIPMed(nn.Module):
    """ViT‑based KCLIP‑Med"""
    def __init__(self, n_cls: int, D: int = 512, top_k: int = 32, T: float = 0.07):
        super().__init__()
        self.visual = timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=0, global_pool="")
        self.proj_v = nn.Linear(self.visual.embed_dim, D, bias=False)
        self.cui_emb = nn.Embedding(n_cls, D)
        self.top_k  = top_k
        self.attn   = nn.MultiheadAttention(D, 8, batch_first=True)
        self.logit_scale = nn.Parameter(torch.log(torch.tensor(1/T)))
    def forward(self, x):
        b = x.size(0)
        tok = self.visual.patch_embed(x)
        cls = self.visual.cls_token.expand(b, -1, -1)
        pos = self.visual.pos_embed[:, : tok.size(1) + 1]
        tok = torch.cat((cls, tok), 1) + pos
        tok = self.visual.blocks(tok)
        tok = self.visual.norm(tok)
        v   = self.proj_v(tok[:, 0])
        cui_tbl = self.cui_emb.weight                              # (C, D)
        sims    = v @ cui_tbl.T
        _, idx  = torch.topk(sims, self.top_k, dim=-1)
        c_k     = self.cui_emb(idx)
        q   = v.unsqueeze(1)
        kv  = torch.cat([q, c_k], 1)
        att, _ = self.attn(q, kv, kv)
        fused   = F.layer_norm(v + att.squeeze(1), (v.size(-1),))
        s = self.logit_scale.exp().clamp(max=100)
        return s * (fused @ cui_tbl.T)

class KCLIPMedSwin(nn.Module):
    def __init__(self, n_cls: int, D: int = 512, top_k: int = 32, T: float = 0.07):
        super().__init__()
        self.visual = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0, global_pool="avg")
        self.proj_v = nn.Linear(self.visual.num_features, D, bias=False)
        self.cui_emb = nn.Embedding(n_cls, D)
        self.top_k = top_k
        self.attn  = nn.MultiheadAttention(D, 8, batch_first=True)
        self.logit_scale = nn.Parameter(torch.log(torch.tensor(1/T)))
    def forward(self, x):
        v_feat = checkpoint.checkpoint(self.visual, x)
        v      = self.proj_v(v_feat)
        cui_tbl = self.cui_emb.weight
        sims    = v @ cui_tbl.T
        _, idx  = torch.topk(sims, self.top_k, dim=-1)
        c_k     = self.cui_emb(idx)
        q   = v.unsqueeze(1)
        kv  = torch.cat([q, c_k], 1)
        att, _ = self.attn(q, kv, kv)
        fused = F.layer_norm(v + att.squeeze(1), (v.size(-1),))
        s = self.logit_scale.exp().clamp(max=100)
        return s * (fused @ cui_tbl.T)

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1)*p)  # Learnable p
        self.eps = eps

    def forward(self, x):
        if x.dim() == 2:  # Handle case where spatial dims are missing
            return x
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1./self.p)

class EfficientNetB1WithFFNN(nn.Module):  # Change name to EfficientNetB1WithFFNN
    def __init__(self, base_model, num_features, num_classes):
        super().__init__()
        self.efficientnet = base_model
        self.efficientnet.classifier = nn.Identity()  # Remove the classifier to get features

        self.gem = GeMPooling()  # GeM pooling to aggregate features

        self.ffnn = nn.Sequential(
            nn.Linear(num_features, 512),  # num_features is 1280 for EfficientNet B1
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),  # num_classes will be the number of unique labels
            nn.Sigmoid()  # Sigmoid activation for multi-label classification
        )

    def forward(self, x):
        x = self.efficientnet(x)  # [B, 1280, H, W]
        if x.dim() == 2:
            x = x.unsqueeze(-1).unsqueeze(-1)  # [B, C] -> [B, C, 1, 1]
        x = self.gem(x)  # Global pooling layer
        x = x.flatten(1)  # Flatten features for FFNN
        return self.ffnn(x)

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 5.  LOAD WEIGHTS                                                        ║
# ╚═══════════════════════════════════════════════════════════════════════╝
chk_root = "/content/drive/MyDrive"               # edit once if your paths differ
EFF_CKPT   = os.path.join(chk_root, "your_model_checkpoint.pth")
EFF1_CKPT   = os.path.join(chk_root, "your_model_checkpoint.pth")
DENSE_CKPT = os.path.join(chk_root, "your_model_checkpoint.pth")
VIT_CKPT   = os.path.join(chk_root, "your_model_checkpoint.pth")
SWIN_CKPT  = os.path.join(chk_root, "your_model_checkpoint.pth")

print("🔗 Loading checkpoints …")

eff   = EfficientNetB0WithFFNN(NUM_C).to(device)
_dense_tmp = torch.load(EFF_CKPT, map_location="cpu")
eff.load_state_dict(_dense_tmp["model_state_dict"])

num_classes = NUM_C
base_model = efficientnet = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.IMAGENET1K_V1)
eff1   = EfficientNetB1WithFFNN(base_model,1280,num_classes).to(device)
eff1_ckpt = torch.load(EFF1_CKPT , map_location='cpu', weights_only=False)
eff1.load_state_dict(eff1_ckpt['model_state_dict'])

dense = build_densenet(NUM_C).to(device)
_dense_tmp = torch.load(DENSE_CKPT, map_location="cpu",weights_only=False)
dense.load_state_dict(_dense_tmp["model_state_dict"])

kclip_vit = KCLIPMed(NUM_C).to(device)
_kc_tmp = torch.load(VIT_CKPT, map_location="cpu",weights_only=False)
kclip_vit.load_state_dict(_kc_tmp["model_state_dict"])

kclip_swin = KCLIPMedSwin(NUM_C).to(device)
_sw_tmp = torch.load(SWIN_CKPT, map_location="cpu",weights_only=False)
kclip_swin.load_state_dict(_sw_tmp["model_state_dict"])

for m in (eff, dense, kclip_vit, kclip_swin):
    m.eval()
print("✅ All models ready\n")

🔗 Loading checkpoints …


Downloading: "https://download.pytorch.org/models/efficientnet_b1_rwightman-bac287d4.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1_rwightman-bac287d4.pth
100%|██████████| 30.1M/30.1M [00:00<00:00, 35.6MB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

✅ All models ready



In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 6.  TEST DATALOADER                                                    ║
# ╚═══════════════════════════════════════════════════════════════════════╝
class TestDS(Dataset):
    def __init__(self, folder: str, tfm):
        self.paths = sorted([os.path.join(folder, p) for p in os.listdir(folder) if p.lower().endswith('.jpg')])
        self.tfm   = tfm
    def __len__(self):  return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert('RGB')
        return self.tfm(img), os.path.basename(p)

test_tfm = T.Compose([T.Resize((224,224)), T.ToTensor()])
TEST_LOADER = DataLoader(TestDS(TEST_FOLDER, test_tfm), batch_size=64, shuffle=False)
print(f"🖼️  Test images: {len(TEST_LOADER.dataset)}\n")

🖼️  Test images: 19267



In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 7.  PREDICT ONCE PER MODEL                                             ║
# ╚═══════════════════════════════════════════════════════════════════════╝
# Mapping: name -> (model, threshold, needs_sigmoid)
MODELS = {
    "EfficientNet": (eff,        0.30, False),  # NN already ends with Sigmoid
    "EfficientNet1": (eff1,        0.28, False),  # NN already ends with Sigmoid
    "DenseNet":     (dense,      0.40, True ),  # raw logits
    "KCLIP-ViT":    (kclip_vit,  0.45, True ),
    "KCLIP-Swin":   (kclip_swin, 0.45, True )
}

pred_store = {}
img_names  = None   # will be captured on first pass

with torch.no_grad():
    for name, (model, thr, need_sig) in MODELS.items():
        preds = []
        names = []
        for xb, nb in tqdm(TEST_LOADER, desc=f"Predicting {name}"):
            xb = xb.to(device)
            out = model(xb)
            if need_sig:
                out = torch.sigmoid(out)
            preds.append((out > thr).cpu().numpy())
            if img_names is None:
                names.extend(nb)  # only once
        pred_store[name] = np.vstack(preds)
    img_names = names
print("✅ Collected predictions for all base models\n")

Predicting KCLIP-Swin:   0%|          | 0/302 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Predicting KCLIP-Swin: 100%|██████████| 302/302 [03:28<00:00,  1.45it/s]

✅ Collected predictions for all base models



In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 8.  LOAD RANKING (ensemble_ranking.csv)                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝
RANK_CSV = "ensemble_ranking.csv"
if not os.path.exists(RANK_CSV):
    raise FileNotFoundError("✘ You need ensemble_ranking.csv in CWD. Run the validation‑ranking step first!")
rank_df = pd.read_csv(RANK_CSV)
rank_df

,models,micro_f1
0,EfficientNet1,0.533386
1,EfficientNet + EfficientNet1,0.533268
2,EfficientNet1 + KCLIP-ViT,0.530518
3,EfficientNet1 + DenseNet,0.530500
4,EfficientNet,0.529411
5,EfficientNet1 + KCLIP-Swin,0.529346
6,EfficientNet + EfficientNet1 + KCLIP-ViT,0.529059
7,EfficientNet + DenseNet,0.528902
8,EfficientNet + EfficientNet1 + DenseNet,0.528641
9,EfficientNet + EfficientNet1 + KCLIP-Swin,0.528137


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# 9.  HELPER TO WRITE A SUBMISSION                                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

def write_submission(preds: np.ndarray, names: list[str], out_path: str):
    ids  = [n.split('.')[0] for n in names]
    cuis = []
    for row in preds:
        cuis.append(';'.join([mlb.classes_[i] for i, v in enumerate(row) if v == 1]))
    pd.DataFrame({"ID": ids, "CUIs": cuis}).to_csv(out_path, index=False)

# ╔═══════════════════════════════════════════════════════════════════════╗
# 10. GENERATE SUBMISSIONS FOR EVERY RANKED ENTRY                        ║
# ╚═══════════════════════════════════════════════════════════════════════╝
SUB_DIR = "/content/submissions"
os.makedirs(SUB_DIR, exist_ok=True)

for rank, row in rank_df.iterrows():
    combo = row["models"].split(" + ")
    if len(combo) == 1:                                  # single model (already computed)
        ensemble_preds = pred_store[combo[0]]
    else:                                                # logical OR of constituent models
        ensemble_preds = np.logical_or.reduce([pred_store[m] for m in combo]).astype(int)

    safe_name = "_".join(c.replace("-","").replace(" ","") for c in combo)
    out_csv   = os.path.join(SUB_DIR, f"submission_rank{rank+1:02d}_{safe_name}.csv")
    write_submission(ensemble_preds, img_names, out_csv)
    print(f"📄 Saved → {out_csv}")

print("\n🎉 All submission files are now in /content/submissions")


📄 Saved → /content/submissions/submission_rank01_EfficientNet1.csv
📄 Saved → /content/submissions/submission_rank02_EfficientNet_EfficientNet1.csv
📄 Saved → /content/submissions/submission_rank03_EfficientNet1_KCLIPViT.csv
📄 Saved → /content/submissions/submission_rank04_EfficientNet1_DenseNet.csv
📄 Saved → /content/submissions/submission_rank05_EfficientNet.csv
📄 Saved → /content/submissions/submission_rank06_EfficientNet1_KCLIPSwin.csv
📄 Saved → /content/submissions/submission_rank07_EfficientNet_EfficientNet1_KCLIPViT.csv
📄 Saved → /content/submissions/submission_rank08_EfficientNet_DenseNet.csv
📄 Saved → /content/submissions/submission_rank09_EfficientNet_EfficientNet1_DenseNet.csv
📄 Saved → /content/submissions/submission_rank10_EfficientNet_EfficientNet1_KCLIPSwin.csv
📄 Saved → /content/submissions/submission_rank11_EfficientNet_KCLIPViT.csv
📄 Saved → /content/submissions/submission_rank12_EfficientNet_KCLIPSwin.csv
📄 Saved → /content/submissions/submission_rank13_EfficientNet1_D

In [ ]:
import shutil

# Define the output zip file name
zip_path = "/content/submissionsV2.zip"

# Create the zip file
shutil.make_archive("/content/submissionsV2", 'zip', "/content/submissions")
print(f"✅ Zipped all submissions to: {zip_path}")

✅ Zipped all submissions to: /content/submissionsV2.zip
